# 22-13 · POST-Redirect-GET

Praktyka do sekcji [„HTML-forms i wysyłanie danych do serwera”

## Cel

Upewnij się, że obsługa formularza odpowiada przekierowaniem (kod 303 See Other), a nie od razu gotową stroną.

## Sprawa robocza

In [ ]:
from flask import Flask, redirect, request, url_for

app = Flask(__name__)
zapisi = []


@app.route("/")
def glavnaya():
    return f"Записей: {len(zapisi)}"


@app.route("/dobavit", methods=["POST"])
def dobavit():
    tekst = request.form.get("tekst", "").strip()
    if tekst:
        zapisi.append(tekst)
    # code=303 (See Other) dokładniejsze niż domyślne przekierowanie Flask (302
    # Found), opisuje POST-Redirect-GET: wynik zobacz inaczej
    # adresować przez GET.
    return redirect(url_for("glavnaya"), code=303)


client = app.test_client()
otvet = client.post("/dobavit", data={"tekst": "Первая запись"})

print("Код ответа:", otvet.status_code)
print("Заголовок Location:", otvet.headers.get("Location"))

## Sprawdzenie wyniku

In [ ]:
assert otvet.status_code == 303
assert otvet.headers.get("Location") is not None
assert zapisi == ["Первая запись"]
print("Верно: POST ответил редиректом 303, а не HTML-страницей напрямую.")

## Zadanie niezależne od zadania ★★

Postępuj ręcznie zgodnie z przekierowaniem: GET adres z nagłówka Location i upewnij się, że pokazuje aktualną liczbę rekordów.

In [ ]:
otvet_posle_redirecta = client.get(otvet.headers["Location"])
telo = otvet_posle_redirecta.get_data(as_text=True)

assert "Записей: 1" in telo
print("Верно: страница после редиректа показывает уже обновлённые данные.")